### Configure API Client

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_SECRET_KEY")

In [ ]:
from binance.client import Client
client = Client(api_key, api_secret, testnet=True)

### Load Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv", index_col='Date', parse_dates=['Date'])
df = df[["Close", "Volume"]].copy()

In [ ]:
df.head()

### Calculating Returns

In [ ]:
import numpy as np

In [ ]:
df["Return"] = df["Close"].div(df["Close"].shift(1))     # This is the return factor from that period's interval
df["Return"] = np.log(df["Return"])    # Converts to log returns for additivity and scale
df

### Contingency Analysis

In [ ]:
import seaborn as sns

In [ ]:
# First cleaning the data to make it suitable for log - no inf, no 0
test_data['Volume'] = test_data['Volume'].replace([np.inf, -np.inf, 0], np.nan)
# test_data.dropna(subset='Volume', inplace=True)    # This was WRONG - never drop na values before using shift, for edge cases
test_data.isna().sum()

In [ ]:
test_data['Vol_ch'] = np.log(test_data_1['Volume'].div(test_data_1['Volume'].shift(1)))

In [ ]:
test_data_1 = test_data.copy()  # Just to create a branch
plt.scatter(x=test_data_1['Vol_ch'], y=test_data_1['Return'])
plt.xlabel("Volume Change")
plt.ylabel("Return")
plt.show()

In [ ]:
# Evidently the data needs to be cleaned to remove outliers - we'll remove the obvious extreme outliers first
minidx = test_data_1['Vol_ch'].idxmin()
maxidx = test_data_1['Vol_ch'].idxmax()

test_data_1.loc[minidx, 'Vol_ch'] = np.nan 
test_data_1.loc[maxidx, 'Vol_ch'] = np.nan

In [ ]:
# Checking the scatter again
plt.scatter(x=test_data_1['Vol_ch'], y=test_data_1['Return'])
plt.xlabel("Volume Change")
plt.ylabel("Return")
plt.show()

In [ ]:
plt.hist(test_data_1['Vol_ch'], bins=100)
plt.show()

Distribution of log changes looks symmetric with no extreme outliers, so I won't remove value ranges. 

**However the approach was quite manual, and therefore trying again with a more statistical approach:**

We will remove extreme outliers using z-score of 3, to catch the 0.3% of extreme values.

In [ ]:
from scipy.stats import skew, kurtosis

test_data_2 = test_data.copy()
test_data_2 = test_data_2.dropna()

# Removing extreme outliers
upper_threshold = np.percentile(test_data_2['Vol_ch'], 99)
lower_threshold = np.percentile(test_data_2['Vol_ch'], 1)

print(upper_threshold)
print(lower_threshold)

test_data_2 = test_data_2.loc[(test_data_2['Vol_ch'] > lower_threshold) & (test_data_2['Vol_ch'] < upper_threshold)]

plt.scatter(x=test_data_2['Vol_ch'], y=test_data_2['Return'])
plt.xlabel("Volume Change")
plt.ylabel("Return")
plt.show()

skewness = skew(test_data_1['Vol_ch'].dropna())
kurt = kurtosis(test_data_1['Vol_ch'].dropna())
print(f"Skewness: {skewness}")
print(f"Kurtosis: {kurt}")

Skew is less around 0.5, kurt less than 2 👍

#### Discretisation 

In [ ]:
test_data = test_data_1.copy()
test_data['Ret_cat'] = pd.qcut(test_data["Return"], q=10, labels=list(range(-5,0)) + list(range(1,6)))
test_data['Vol_cat'] = pd.qcut(test_data["Vol_ch"], q=10, labels=list(range(-5,0)) + list(range(1,6)))

In [ ]:
# Contingency Table show distribution for each combination of ret/vol categories
matrix = pd.crosstab(test_data['Ret_cat'], test_data['Vol_cat'])
matrix

In [ ]:
plt.figure(figsize=(12,8))
ax = sns.heatmap(matrix, cmap='RdYlBu_r')
ax.invert_yaxis()

### Consolidated Script

In [ ]:
# Load data
data = pd.read_csv("../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv", index_col='Date', parse_dates=['Date'])
data = data[["Close", "Volume"]].copy()

data["Return"] = data["Close"].div(data["Close"].shift(1))     # This is the return factor from that period's interval
data["Return"] = np.log(data["Return"])    # Converts to log returns for additivity and scale

# First cleaning the data to make it suitable for log - no inf, no 0
data['Volume'] = data['Volume'].replace([np.inf, -np.inf, 0], np.nan)
print(f"NA values: {data.isna().sum()}")

data['Vol_ch'] = np.log(data['Volume'].div(data['Volume'].shift(1)))

print(f"NA values: {data.isna().sum()}")


# Removing extreme outliers
data = data.dropna()
upper_threshold = np.percentile(data['Vol_ch'], 99)
lower_threshold = np.percentile(data['Vol_ch'], 1)

data = data.loc[(data['Vol_ch'] > lower_threshold) & (data['Vol_ch'] < upper_threshold)]

plt.scatter(x=data['Vol_ch'], y=data['Return'])
plt.xlabel("Volume Change")
plt.ylabel("Return")
plt.show()

data['Ret_cat'] = pd.qcut(data["Return"], q=10, labels=list(range(-5,0)) + list(range(1,6)))
data['Vol_cat'] = pd.qcut(data["Vol_ch"], q=10, labels=list(range(-5,0)) + list(range(1,6)))

matrix = pd.crosstab(data['Ret_cat'], data['Vol_cat'])
display(matrix)

plt.figure(figsize=(12,8))
ax = sns.heatmap(matrix, cmap='RdYlBu_r')
ax.invert_yaxis()

### Refinements
This doesn't feel very insightful though - since it's just the return and volume change at the same time. It would be better to shift it such to see subsequent returns AFTER big volume changes, for example

## Learnings
1. Never use dropna() before using a shift feature, otherwise the boundary values will be wrong when shifting